# Weather Prediction Project: Notebook 6. Final Model Summary

This notebook is the final synthesis of the modeling work. The exploration stage has already been completed, the ranking stage has retained a baseline and a stronger final model, and the refinement stage has locked the final configuration. The task here is therefore not to introduce major new experiments, but to summarize the retained models, state the final recommendation clearly, and draw the main lessons from the project.

## Step 1. Load the final saved evidence

The closing notebook works directly from the retained winner package so that the final narrative remains aligned with the saved summaries, challenger comparisons, calibration results, robustness tables, and SHAP outputs.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

possible_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "models").exists() and (path / "notebooks").exists()
    ),
    Path.cwd().resolve(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.visualization.final_report_charts import (
    build_calibration_curve_chart,
    build_location_robustness_chart,
    build_model_ranking_chart,
    build_robustness_time_chart,
    build_segment_bar_chart,
)

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_colwidth = 180

WINNER_DIR = PROJECT_ROOT / "models" / "final_winner_package"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
THRESHOLD_FIGURE_PATH = FIGURES_DIR / "fig_68_final_winner_threshold_curve.png"
SHAP_BAR_PATH = FIGURES_DIR / "fig_54_final_hybrid_shap_bar.png"
SHAP_BEESWARM_PATH = FIGURES_DIR / "fig_55_final_hybrid_shap_beeswarm.png"
SHAP_WATERFALL_PATH = FIGURES_DIR / "fig_56_final_hybrid_shap_waterfall.png"


def display_figure(title: str, path: Path, width: int = 1120) -> None:
    display(Markdown(f"**{title}**"))
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"_Missing figure: {path.name}_"))


final_results = pd.read_csv(WINNER_DIR / "final_model_comparison.csv")
threshold_curve = pd.read_csv(WINNER_DIR / "baseline_threshold_curve.csv")
calibration_methods = pd.read_csv(WINNER_DIR / "winner_calibration_methods.csv")
raw_curve = pd.read_csv(WINNER_DIR / "winner_uncalibrated_curve.csv")
calibrated_curve = pd.read_csv(WINNER_DIR / "winner_climate_regime_isotonic_curve.csv")
shap_importance = pd.read_csv(WINNER_DIR / "final_model_shap_importance.csv")
robustness_by_season = pd.read_csv(WINNER_DIR / "robustness_by_season.csv")
robustness_by_climate = pd.read_csv(WINNER_DIR / "robustness_by_climate_regime.csv")
robustness_by_location = pd.read_csv(WINNER_DIR / "robustness_by_location.csv")
robustness_time_windows = pd.read_csv(WINNER_DIR / "robustness_time_windows.csv")

with open(WINNER_DIR / "final_hybrid_refinement_summary.json", "r", encoding="utf-8") as file:
    refinement_summary = json.load(file)

with open(WINNER_DIR / "winner_calibration_summary.json", "r", encoding="utf-8") as file:
    calibration_summary = json.load(file)

with open(WINNER_DIR / "dense_nn_summary.json", "r", encoding="utf-8") as file:
    dense_summary = json.load(file)

with open(WINNER_DIR / "lstm_summary.json", "r", encoding="utf-8") as file:
    lstm_summary = json.load(file)

with open(WINNER_DIR / "gru_summary.json", "r", encoding="utf-8") as file:
    gru_summary = json.load(file)

with open(WINNER_DIR / "attention_summary.json", "r", encoding="utf-8") as file:
    attention_summary = json.load(file)

with open(WINNER_DIR / "robustness_summary.json", "r", encoding="utf-8") as file:
    robustness_summary = json.load(file)

with open(WINNER_DIR / "retained_logistic_baseline_summary.json", "r", encoding="utf-8") as file:
    logistic_summary = json.load(file)

final_results["rank"] = pd.to_numeric(final_results["rank"], errors="coerce")
winner_row = final_results.loc[final_results["rank"] == 1].iloc[0]
cnn_row = final_results.loc[final_results["model_view"] == "cnn_14_dilated"].iloc[0]
dense_best = dense_summary["best_result"]
lstm_best = lstm_summary["best_result"]
gru_best = gru_summary["best_result"]
attention_best = attention_summary["best_result"]
recommended_calibrated = calibration_summary["recommended_calibrated"]

**Interpretation.** The final notebook is deliberately synthetic. By this point the workflow has already explored, ranked, and refined; the task now is to state clearly what was retained and why. Reading directly from the saved winner package prevents a last-minute change of narrative and keeps the conclusion anchored to the same evidence used throughout the later workflow.

This matters for credibility. A final notebook should not feel like another experiment. It should feel like the place where the modeling argument is assembled in its clearest form.

<!-- INTEGRATED_SUMMARY_MERGE -->
### How the integrated studies become one final project

The final recommendation is not based on one notebook winning over another. The project combines the strongest evidence from the full workflow and assigns each integrated study the role it supports best.

| Integrated study | Concrete retained contribution | Where it shows up in the final notebook path |
| --- | --- | --- |
| Climate, rainfall-zone, and atmospheric-momentum audit | Establishes imbalance, structured missingness, heterogeneous rainfall seasons, and the need for climate-aware features. | Notebooks 1 and 2 |
| Broader preprocessing study | Reinforces chronological forecasting splits across the shared station network, variable-specific missingness logic, circular encoding, and the use of richer all-feature tables for selected families. | Notebook 2 |
| Automated classifier sweep | Shows that the early chronological LazyPredict pass surfaced Nearest Centroid, XGBoost, SVC, and Logistic Regression as families worth explicit reruns rather than leaving the wider search space implicit; its aggregate default-label metrics remain triage evidence, not direct ranking evidence. | Notebooks 3 and 4 |
| Boosted-tree and calibration studies | Show XGBoost and LightGBM near `ROC-AUC 0.89`, F1 around `0.67`, threshold tuning near `0.30`, and provide feature-importance, SHAP, threshold, and calibration evidence. | Notebooks 3, 4, and 5 |
| Static deep architecture coverage | Shows that sequential dense, Wide & Deep, and ResNet variants were all tested on the scaled high-dimensional table, while TabNet was explicitly scoped as an attention-style tabular deep-learning prototype rather than promoted into the ranked challenger ladder. | Notebooks 3 and 4 |
| Deep and temporal challengers | Show that the best alternatives cluster just below the winner instead of collapsing, which makes the final CatBoost selection more credible. | Notebooks 3, 4, and 6 |
| Final CatBoost refinement | Locks the strongest final classifier after threshold, calibration, robustness, and interpretability checks. | Notebooks 5 and 6 |

This is why the final answer is coherent. Logistic Regression is retained as the interpretable benchmark. CatBoost is retained as the final classifier. The boosted-tree calibration studies, the automated sweep, and the broader deep-learning studies remain visible as serious supporting evidence that sharpened the final logic around preprocessing, thresholding, calibration, and interpretation. The result should read as one group project with one modeling argument, not as parallel notebooks left unresolved.

<!-- FINAL_RANKING_SURFACES -->
### Unified ranking and feature-space summary

The easiest way to read the final project is as one ranking ladder with explicit feature surfaces rather than as separate notebook islands. The table below is the final cross-notebook map that ties notebook 4's broad ranking to notebook 5's controlled refinement.

| Final project standing | Models or studies | Unified feature surface description | What survives into the final recommendation |
| --- | --- | --- | --- |
| Retained final classifier | CatBoost raw classification view | 68-feature hybrid-plus-core winner representation | Final binary rain or no-rain recommendation. |
| Retained calibrated probability view | CatBoost climate-regime isotonic calibration | the same 68-feature hybrid-plus-core winner representation with time-aware calibration | Probability-reporting view, not a new classification winner. |
| Closest non-retained challengers | Temporal LSTM, GRU, Attention, best CNN, and the 83-feature geo-context challenger | rolling 68-driver sequence views or the controlled 83-feature geo-context extension | Show that the winner survived credible near ties. |
| Strong merged boosted-tree challengers | LightGBM, calibrated LightGBM, all-feature XGBoost, calibrated XGBoost, and aligned XGBoost | broader all-feature weather table with climate context or the aligned top-25 baseline table | Supply challenger evidence and the calibration, threshold, SHAP, and feature-importance lessons carried into the final workflow. |
| Static deep challengers | saved dense benchmark, sequential dense, Wide & Deep, and ResNet | 68-feature scaled high-dimensional hybrid representation or the broader scaled high-dimensional table | Demonstrate that neural architectures were tried seriously through scored static deep runs, even though they were not retained. |
| Interpretable benchmark | Logistic Regression | aligned top-25 feature table | Reference benchmark kept for explainability and accountability. |
| Exploratory side studies | LazyPredict automated sweep, Random Forest, KNN, Decision Tree, Linear SVM, Nearest Centroid, TPOT, and clustering diagnostics | surfaces documented in notebooks 2 and 3 | Documented search space, not retained final models. |

TabNet remains visible in the project as documented prototype coverage, but it sits outside this ranked ladder because the repository does not retain a scored TabNet result package.

That table is the unifying rule across notebooks 4 to 6. Notebook 4 ranks the full selection-stage search space. Notebook 5 tests only controlled refinements on the locked CatBoost base plus its calibrated and geo-context variants. Notebook 6 reports the final decision without losing sight of the broader merged ranking.

## Step 2. Compare the retained final model against the interpretable baseline

The final recommendation is stronger when it is stated relative to a credible baseline rather than in isolation. The table below therefore keeps both retained models visible: Logistic Regression as the interpretable benchmark and CatBoost as the stronger final predictive model.

In [ ]:
baseline_result = refinement_summary["baseline_result"]
logistic_holdout = logistic_summary["holdout_metrics"]

baseline_vs_final = pd.DataFrame(
    [
        {
            "retained_role": "Interpretable baseline",
            "model": "Logistic Regression",
            "forecast_target": "RainTomorrow",
            "probability_of_rain": "p_rain",
            "probability_of_no_rain": "1 - p_rain",
            "representative_holdout_roc_auc": logistic_holdout["roc_auc"],
            "representative_holdout_f1": logistic_holdout["f1"],
            "representative_holdout_recall": logistic_holdout["recall"],
            "main_reason_for_retention": "Keeps the project anchored to a transparent and reproducible benchmark model.",
        },
        {
            "retained_role": "Final predictive model",
            "model": "CatBoost",
            "forecast_target": "RainTomorrow",
            "probability_of_rain": "p_rain",
            "probability_of_no_rain": "1 - p_rain",
            "representative_holdout_roc_auc": baseline_result["test_roc_auc"],
            "representative_holdout_f1": baseline_result["test_f1"],
            "representative_holdout_recall": baseline_result["test_recall"],
            "main_reason_for_retention": "Provides the strongest overall balance between predictive strength and methodological fit.",
        },
    ]
)

display(baseline_vs_final)


**Interpretation.** The retained baseline and the retained final model serve different purposes, and the table makes that distinction visible. Logistic Regression reaches holdout ROC-AUC 0.8620, F1 0.6300, and recall 0.6858. Those numbers are not trivial; they show that a transparent linear benchmark already captures meaningful rainfall structure. CatBoost then raises the benchmark to ROC-AUC 0.9016, F1 0.6853, and recall 0.7510.

This comparison matters because it makes the gain interpretable. The final model is not just better in an abstract sense; it buys roughly 5.5 F1 points and about 6.5 recall points over a transparent baseline while staying on the same forecasting target. These are post-exploration holdout summaries rather than single-use pristine test readings, which is why the later robustness section carries more weight for the final reliability claim.


## Step 3. Keep the locked refinement shortlist visible, but secondary

This shortlist is intentionally narrower than notebook 4's broader cross-family ranking. It shows the challengers that remained active once the winner package moved from selection into controlled refinement, not every earlier challenger that was still important to the overall project story.


In [ ]:
shortlisted_field = final_results[
    [
        "rank",
        "display_label",
        "selection_basis",
        "selection_threshold",
        "test_roc_auc",
        "test_f1",
        "test_recall",
    ]
].rename(
    columns={
        "display_label": "candidate_view",
        "selection_basis": "selection_rule",
        "selection_threshold": "decision_threshold",
        "test_roc_auc": "holdout_roc_auc",
        "test_f1": "holdout_f1",
        "test_recall": "holdout_recall",
    }
)

display(shortlisted_field)
display(Markdown("**App-style best-model comparison**"))
display(build_model_ranking_chart(final_results))


**Interpretation.** This chart is the locked refinement shortlist rather than the broader cross-family shortlist. Within that narrower late-stage field, the best retained alternatives are not far away: the Temporal CNN reaches F1 0.6824, the geo-context challenger 0.6823, and the calibrated climate-regime isotonic view 0.6803, all while staying near the same ROC-AUC band. This is exactly what a serious project should look like: the winner is not isolated by a huge margin, but it still finishes first after several credible attempts to displace it.

That reading makes the CatBoost recommendation more robust. If the nearest challengers had failed badly, the final claim would be weaker because the search space would look shallow. Instead, the workflow shows that several smart alternatives were explored and still did not beat the locked winner on the full decision balance.

The broader cross-family shortlist is still visible elsewhere in the notebook set. Notebook 4 and the unified ranking table above keep the expanded-missingness CatBoost challenger and the retained Temporal LSTM visible as important non-retained alternatives, but they are not the same thing as this locked refinement shortlist.


<!-- SHORTLIST_SCOPE_NOTE -->
The shortlist chart in notebook 6 is intentionally narrower than notebook 4's merged ranking table. It shows the saved winner-package challengers that remained active once refinement started. The broader LightGBM, calibrated XGBoost, aligned XGBoost, Logistic Regression, and broader static deep-learning studies are still part of the final story, but they belong to notebook 4's selection-stage ranking and the unified ranking table above rather than to the refinement shortlist chart itself.

## Step 4. Separate classification strength from probability reliability

The final synthesis should keep the distinction between classification quality and calibration explicit. The raw CatBoost view remains the best classifier, while the calibrated view matters when the project needs the probabilities themselves to be as trustworthy as possible.

In [ ]:
raw_view = calibration_summary["uncalibrated"]
calibrated_view = calibration_summary["recommended_calibrated"]

classification_vs_probability = pd.DataFrame(
    [
        {
            "model_view": "Raw CatBoost classification view",
            "decision_threshold": raw_view["validation_threshold"],
            "holdout_roc_auc": raw_view["test_roc_auc"],
            "holdout_f1": raw_view["test_f1"],
            "holdout_brier": raw_view["test_brier"],
            "holdout_log_loss": raw_view["test_log_loss"],
        },
        {
            "model_view": calibrated_view["method_label"],
            "decision_threshold": calibrated_view["validation_threshold"],
            "holdout_roc_auc": calibrated_view["test_roc_auc"],
            "holdout_f1": calibrated_view["test_f1"],
            "holdout_brier": calibrated_view["test_brier"],
            "holdout_log_loss": calibrated_view["test_log_loss"],
        },
    ]
)

display(classification_vs_probability)
display(Markdown("**Decision-threshold view**"))
display(Markdown("_The dashed vertical line marks the retained validation-selected threshold._"))
display_figure("Final winner threshold sweep", THRESHOLD_FIGURE_PATH)
display(Markdown("**App-style calibration view**"))
display(build_calibration_curve_chart(raw_curve, calibrated_curve))

**Interpretation.** One of the clearest lessons of the workflow is that classification strength and probability reliability are related but different objectives. The raw CatBoost winner is the best classifier. The calibrated climate-regime isotonic view, however, improves Brier score and log loss substantially, giving better-behaved probability estimates even though F1 slips slightly from 0.6853 to 0.6803.

The probability-view comparison keeps the calibration-study threshold of `0.56` for the raw row so it stays directly comparable with the calibrated row at `0.34`; the locked classification recommendation from notebook 5 still uses `0.58`. That distinction is methodologically mature. It prevents the project from pretending that one metric answers every question. When the task is a rain or no-rain decision, the raw winner is preferred. When the task is to communicate trustworthy rainfall probabilities, the calibrated view becomes the better output.

## Step 5. Summarize what added complexity helped and what did not

The project tested several forms of added complexity, including deeper nonlinear models, temporal sequence models, contextual extensions, and calibration layers. The final notebook should therefore state clearly which complexity proved useful and which did not improve generalization enough to change the conclusion.

In [ ]:
deep_challengers = pd.DataFrame(
    [
        {
            "model_family": "Dense neural network",
            "representative_holdout_roc_auc": dense_best["test_roc_auc"],
            "representative_holdout_f1": dense_best["test_f1"],
            "main_reading": "Competitive static deep challenger, but still weaker than the retained CatBoost winner.",
        },
        {
            "model_family": "Temporal LSTM",
            "representative_holdout_roc_auc": lstm_best["test_roc_auc"],
            "representative_holdout_f1": lstm_best["test_f1"],
            "main_reading": "Strong temporal challenger, showing that recent history matters, but not enough to replace CatBoost.",
        },
        {
            "model_family": "Temporal GRU",
            "representative_holdout_roc_auc": gru_best["test_roc_auc"],
            "representative_holdout_f1": gru_best["test_f1"],
            "main_reading": "Another strong recurrent alternative, especially on recall, but not the cleanest final choice.",
        },
        {
            "model_family": "Temporal Attention",
            "representative_holdout_roc_auc": attention_best["test_roc_auc"],
            "representative_holdout_f1": attention_best["test_f1"],
            "main_reading": "Close on ranking quality, but still slightly weaker on the full decision balance.",
        },
        {
            "model_family": "Geo-context extension",
            "representative_holdout_roc_auc": final_results.loc[
                final_results["model_view"] == "geo_context_challenger", "test_roc_auc"
            ].iloc[0],
            "representative_holdout_f1": final_results.loc[
                final_results["model_view"] == "geo_context_challenger", "test_f1"
            ].iloc[0],
            "main_reading": "Scientifically useful extension, but not a clean enough gain to replace the simpler winner.",
        },
    ]
)

display(deep_challengers)

**Interpretation.** The model comparison shows that extra complexity helped most when it improved representation rather than when it simply increased architectural sophistication. The decisive step was the move from the aligned top-25 table to the 68-feature hybrid representation. That step let CatBoost absorb missingness signals, lag changes, seasonal geometry, and moisture-interaction structure in a way that materially improved final performance.

By contrast, several more elaborate alternatives remained scientifically useful but not decisively better. The dense network was competitive but too recall-heavy. The recurrent and attention models showed that temporal history matters, yet they did not produce a clean enough advantage to justify replacing the stronger tabular winner. The lesson is not that complexity failed; it is that the right kind of complexity was representational rather than architectural.

## Step 6. Close with robustness evidence

A final recommendation should end with visible evidence that the winner is stable beyond a single holdout slice. The next block keeps the temporal and segmented robustness views together so the final claim stays tied to consistency across time, seasons, climate regimes, and locations.

In [ ]:
robustness_headline = pd.DataFrame(
    [
        {
            "rolling_split_count": int(len(robustness_time_windows)),
            "mean_holdout_f1": float(robustness_time_windows["test_f1"].mean()),
            "sample_std_holdout_f1": float(robustness_time_windows["test_f1"].std(ddof=1)),
            "mean_holdout_roc_auc": float(robustness_time_windows["test_roc_auc"].mean()),
            "sample_std_holdout_roc_auc": float(robustness_time_windows["test_roc_auc"].std(ddof=1)),
        }
    ]
)

season_view = robustness_by_season[
    ["segment_value", "support", "event_rate", "roc_auc", "f1", "precision", "recall", "eligible_for_summary"]
].sort_values("f1", ascending=False)
climate_view = robustness_by_climate[
    ["segment_value", "support", "event_rate", "roc_auc", "f1", "precision", "recall", "eligible_for_summary"]
].sort_values("f1", ascending=False)
location_view = robustness_by_location[
    ["segment_value", "climate_regime", "support", "event_rate", "roc_auc", "f1", "precision", "recall", "eligible_for_summary"]
].sort_values("f1", ascending=False)

display(robustness_headline)
display(Markdown("**App-style temporal robustness chart**"))
display(build_robustness_time_chart(robustness_time_windows))
display(Markdown("**App-style season-level robustness chart**"))
display(build_segment_bar_chart(season_view, "Season-level F1"))
display(Markdown("**App-style climate-regime robustness chart**"))
display(build_segment_bar_chart(climate_view, "Climate-regime F1"))
display(Markdown("**App-style location-level robustness chart**"))
display(build_location_robustness_chart(location_view))


**Interpretation.** The robustness block is the last quality-control step before the final recommendation. Across rolling windows, the winner keeps mean holdout F1 0.6800 with sample standard deviation 0.0098 and mean ROC-AUC 0.9037 with sample standard deviation 0.0041, so the headline result is not dependent on one unusually favorable period. The segmented views are equally useful because they show where the model is strongest and where it becomes harder: Winter remains the easiest season, Summer is weaker, and Arid regimes are more difficult than Winter-dominant ones, but none of these slices indicates a collapse of the forecasting logic.

That matters more than a small leaderboard gain because it shows the final model generalizes with believable variation rather than unstable behavior. The recommendation is therefore supported not only by one top score, but by repeated evidence that the winner holds up across time and across operational contexts.

## Step 7. Interpret the final winner with SHAP

The final summary should also explain why the retained model is making its decisions. The next block therefore isolates the SHAP evidence instead of leaving it buried inside the robustness section, so the explanation figures are easy to find and compare with the app.

In [ ]:
top_shap = shap_importance.head(8)[["feature", "mean_abs_shap"]].rename(
    columns={"feature": "leading_driver", "mean_abs_shap": "mean_absolute_shap"}
)

display(top_shap)
display_figure("SHAP bar view of the final winner", SHAP_BAR_PATH)
display_figure("SHAP beeswarm view of the final winner", SHAP_BEESWARM_PATH)
display_figure("SHAP waterfall illustration for one prediction", SHAP_WATERFALL_PATH)


**Interpretation.** The SHAP ranking shows that the final winner is grounded in sensible weather structure. Sunshine, wind gust speed, the cloud-humidity interaction at 3 PM, pressure at 3 PM, and rainfall dominate the explanation pattern, which is consistent with how late-day atmospheric instability, moisture retention, frontal pressure behavior, and recent precipitation shape next-day rain probability. In other words, the model is not only separating classes well; it is doing so through drivers that make physical and forecasting sense.

This interpretability layer is especially important because the project retained CatBoost over more visibly complex challengers such as sequence models. The final model does not win by being the most sophisticated architecture on paper. It wins because it combines strong predictive performance, temporal stability, and an explanation profile that remains coherent with the meteorological task.

## Integrated Project Synthesis

The final notebook now represents the combined project rather than one isolated notebook track. The audit contributed the data-quality and forecasting constraints; preprocessing translated those constraints into a reproducible feature table; the broader exploratory studies widened the feature-selection, scaling, XGBoost, LightGBM, SVC, clustering, and dense-neural search; the ranking notebook brought those results back to one decision rule; and the refinement notebook locked the final operating setup.

| Project layer | What the merged notebooks contribute |
| --- | --- |
| Data audit | Missingness structure, class imbalance, temporal dependence, station heterogeneity, climate-zone patterns, and meteorological feature intuition. |
| Preprocessing | Chronological splitting, canonical feature names, lag/difference features, climate and location context, feature ranking, and study-specific scaling where justified. |
| Broad modeling | Linear, tree, boosted, distance, margin, LightGBM, SVC, clustering, sequence, CNN, attention, and dense-neural challengers. |
| Selection | A role-based decision that keeps Logistic Regression as the interpretable baseline and CatBoost as the final predictive winner. |
| Refinement | Threshold tuning, hyperparameter review, feature-extension checks, calibration, robustness windows, segment diagnostics, and SHAP interpretation. |

That is the clean group story: the project explored widely, then narrowed deliberately. The final recommendation is not based on one isolated leaderboard cell; it is based on a shared sequence of evidence that stays consistent from raw data audit to final model explanation. Because the common holdout was already viewed during exploration, the strongest reliability claim comes from the rolling-window and segment-robustness evidence rather than from a single untouched test-block narrative.

## Final Conclusion

The Weather Prediction Project resolves into a clear decision chain rather than a collection of disconnected trials. Exploration established that linear benchmarks were useful but limited, boosted trees were the strongest tabular family, and temporal and dense neural models were credible challengers. Ranking then retained two models with different purposes: Logistic Regression for interpretability and CatBoost for predictive strength. Refinement finally showed that the locked CatBoost configuration was not meaningfully improved by retuning or by more complex feature and context variants, while calibration and robustness analysis clarified how its outputs should be interpreted.

The project therefore ends with a defensible pair and a clear recommendation. Logistic Regression remains the reference model because it keeps the rainfall task explainable. CatBoost remains the final predictive model because it offers the best balance between discrimination, recall, temporal robustness, and methodological suitability for forecasting `RainTomorrow`. That conclusion is stronger because it was reached step by step, under time-aware validation, with threshold tuning confined to validation data inside each study and with the rolling-window evidence carrying more methodological weight than a single post-exploration holdout score. Because location remains an active predictor and parts of the preprocessing use same-date cross-station context, the cleanest deployment claim is forecasting within the observed station network rather than zero-shot transfer to entirely unseen stations.